# Export completo dei ranking cane–famiglia

Questo notebook calcola il ranking completo dei cani per uno specifico profilo famiglia e salva il risultato in un file CSV.

A differenza del notebook `05_matching_experiments.ipynb`, qui non viene mostrata soltanto la Top 10: vengono esportati **tutti i cani del dataset**, con:

- compatibilità strutturata;
- similarità semantica BERT;
- score finale;
- dati principali del cane;
- descrizione testuale dell'annuncio.

Il file CSV prodotto può essere aperto con Excel o utilizzato nella relazione finale.


## 1. Import delle librerie

In [21]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

## 2. Caricamento dei dati

Vengono caricati:

- il dataset pulito dei cani;
- gli embedding BERT già calcolati nel notebook 03;
- il file `family_profiles.json`, che contiene diversi profili famiglia.


In [22]:
dogs = pd.read_csv("../data/processed/dogs_clean.csv")
bert_embeddings = np.load("../data/processed/bert_embeddings.npy")

profiles_path = Path("../data/family_profiles.json")

with open(profiles_path, "r", encoding="utf-8") as f:
    family_profiles = json.load(f)

print("Dataset cani:", dogs.shape)
print("Embedding BERT:", bert_embeddings.shape)
print("Profili famiglia disponibili:")
for profile_id in family_profiles.keys():
    print("-", profile_id)

Dataset cani: (8132, 30)
Embedding BERT: (8132, 768)
Profili famiglia disponibili:
- family_children_apartment
- single_active_person
- senior_applicant
- experienced_countryside_family
- first_time_adopter


## 3. Scelta del profilo famiglia

Modifica il valore di `selected_profile_id` per scegliere su quale famiglia calcolare il ranking.

Profili disponibili:

- `family_children_apartment`
- `single_active_person`
- `senior_applicant`
- `experienced_countryside_family`
- `first_time_adopter`


In [23]:
selected_profile_id = "family_children_apartment"

family_profile = family_profiles[selected_profile_id]

family_profile

{'profile_name': 'Famiglia con bambini in appartamento',
 'applicant_age': '31-60',
 'children': True,
 'dog_experience': 'some',
 'housing': 'apartment',
 'garden': False,
 'preferred_gender': 'indifferent',
 'preferred_age': 'young',
 'preferred_size': 'small',
 'preferred_fur': 'short',
 'daily_time': '2-4',
 'activity_level': 'moderate',
 'ideal_dog_description': 'We are looking for a small, affectionate and friendly dog suitable for a family with children. The dog should be calm at home, sociable, easy to manage and suitable for apartment life.'}

## 4. Regole di compatibilità strutturata

La compatibilità strutturata confronta le preferenze della famiglia con le caratteristiche del cane.

Il sistema distingue tra:

- vincoli hard, che possono portare a esclusione o punteggio pari a 0;
- vincoli soft, che riducono o aumentano il punteggio.

Regole principali:

- meno di 1 ora al giorno disponibile: adozione non consigliata;
- cane extra large senza giardino: esclusione;
- nessuna esperienza con cane extra large: esclusione;
- nessuna esperienza con cane large: forte penalizzazione;
- cane large senza giardino: forte penalizzazione;
- richiedente over 60: cuccioli penalizzati, adulti e anziani favoriti.


In [24]:
def match_preference(preference, dog_value):
    if preference == "indifferent":
        return 1.0
    return 1.0 if preference == dog_value else 0.0


def size_similarity(preferred_size, dog_size):
    if preferred_size == "indifferent":
        return 1.0

    size_order = {
        "small": 0,
        "medium": 1,
        "large": 2,
        "extra_large": 3
    }

    if preferred_size not in size_order or dog_size not in size_order:
        return 0.0

    distance = abs(size_order[preferred_size] - size_order[dog_size])

    if distance == 0:
        return 1.0
    elif distance == 1:
        return 0.5
    else:
        return 0.0


def compute_structured_score(dog, family):
    # Hard constraint: less than 1 hour per day is not compatible with adoption.
    if family["daily_time"] == "<1":
        return 0.0

    # Hard constraint: extra large dogs require a garden.
    if dog["maturity_size_label"] == "extra_large" and not family["garden"]:
        return 0.0

    # Hard constraint: no dog experience and extra large dog.
    if family["dog_experience"] == "none" and dog["maturity_size_label"] == "extra_large":
        return 0.0

    scores = []

    # Direct preferences
    scores.append(match_preference(family["preferred_gender"], dog["gender_label"]))
    scores.append(match_preference(family["preferred_age"], dog["age_group"]))
    scores.append(size_similarity(family["preferred_size"], dog["maturity_size_label"]))
    scores.append(match_preference(family["preferred_fur"], dog["fur_length_label"]))

    # Children rule
    if family["children"]:
        scores.append(1.0 if dog["age_group"] in ["puppy", "young"] else 0.5)

    # Garden and size rule
    if dog["maturity_size_label"] == "large":
        scores.append(1.0 if family["garden"] else 0.2)
    elif dog["maturity_size_label"] == "extra_large":
        scores.append(1.0)
    else:
        scores.append(1.0)

    # Time availability rule
    if family["daily_time"] == "1-2":
        scores.append(0.6 if dog["age_group"] == "puppy" else 0.8)
    else:
        scores.append(1.0)

    # Experience rule
    if family["dog_experience"] == "none":
        if dog["maturity_size_label"] == "large":
            scores.append(0.2)
        elif dog["age_group"] == "puppy":
            scores.append(0.5)
        else:
            scores.append(1.0)
    else:
        scores.append(1.0)

    # Older applicant rule
    if family["applicant_age"] == "over_60":
        scores.append(1.0 if dog["age_group"] in ["adult", "senior"] else 0.4)

    return np.mean(scores)

## 5. Calcolo dello score strutturato per tutti i cani

La funzione viene applicata a tutti gli 8132 cani del dataset.


In [25]:
dogs_export = dogs.copy()

dogs_export["structured_score"] = dogs_export.apply(
    lambda row: compute_structured_score(row, family_profile),
    axis=1
)

print("Numero totale cani:", len(dogs_export))
print("Cani esclusi dai vincoli hard:", (dogs_export["structured_score"] == 0).sum())
print("Cani valutati:", (dogs_export["structured_score"] > 0).sum())

dogs_export[[
    "Name",
    "age_group",
    "gender_label",
    "maturity_size_label",
    "fur_length_label",
    "structured_score"
]].head()

Numero totale cani: 8132
Cani esclusi dai vincoli hard: 22
Cani valutati: 8110


,Name,age_group,gender_label,maturity_size_label,fur_length_label,structured_score
0,Brisco,puppy,male,medium,medium,0.6875
1,Miko,puppy,female,medium,short,0.8125
2,Hunter,puppy,male,medium,short,0.8125
3,Siu Pak & Her 6 Puppies,puppy,female,medium,short,0.8125
4,Bear,puppy,male,medium,short,0.8125


## 6. Similarità semantica tramite BERT

Il testo libero della famiglia viene trasformato in embedding BERT e confrontato con gli embedding delle descrizioni dei cani.

Gli embedding dei cani sono già stati calcolati nel notebook 03 e vengono riutilizzati per evitare di ricalcolarli.


In [26]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)
bert_model.eval()

print("BERT model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT model loaded.


In [27]:
def get_bert_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = bert_model(**inputs)

    embedding = outputs.last_hidden_state.mean(dim=1)
    return embedding.squeeze().numpy()


family_embedding = get_bert_embedding(family_profile["ideal_dog_description"])

semantic_scores = cosine_similarity(
    family_embedding.reshape(1, -1),
    bert_embeddings
).flatten()

# Normalizzazione della cosine similarity da [-1, 1] a [0, 1]
dogs_export["bert_similarity"] = (semantic_scores + 1) / 2

dogs_export[["Name", "bert_similarity"]].head()

,Name,bert_similarity
0,Brisco,0.907398
1,Miko,0.876957
2,Hunter,0.925550
3,Siu Pak & Her 6 Puppies,0.822069
4,Bear,0.827598


## 7. Calcolo dello score finale

Lo score finale combina:

- 70% compatibilità strutturata;
- 30% similarità semantica BERT.


In [28]:
dogs_export["final_score"] = (
    0.7 * dogs_export["structured_score"] +
    0.3 * dogs_export["bert_similarity"]
)

dogs_export[[
    "Name",
    "structured_score",
    "bert_similarity",
    "final_score"
]].head()

,Name,structured_score,bert_similarity,final_score
0,Brisco,0.6875,0.907398,0.753470
1,Miko,0.8125,0.876957,0.831837
2,Hunter,0.8125,0.925550,0.846415
3,Siu Pak & Her 6 Puppies,0.8125,0.822069,0.815371
4,Bear,0.8125,0.827598,0.817029


## 8. Creazione del ranking completo

Il ranking viene ordinato in modo decrescente in base allo score finale.

Il DataFrame contiene tutti i cani del dataset, non solo la Top 10.


In [ ]:
export_columns = [

    "structured_score",
    "bert_similarity",
    "final_score",
    "PetID",
    "Name",
    "Age",
    "age_group",
    "gender_label",
    "maturity_size_label",
    "fur_length_label",
    "Vaccinated",
    "Dewormed",
    "Sterilized",
    "Health",
    "Fee",
    "PhotoAmt",
    "Description"

]

full_ranking = dogs_export.sort_values(
    by="final_score",
    ascending=False
)[export_columns]

print("Numero cani nel ranking completo:", len(full_ranking))

full_ranking.head(20)

Numero cani nel ranking completo: 8132


,PetID,Name,Age,age_group,gender_label,maturity_size_label,fur_length_label,Vaccinated,Dewormed,Sterilized,Health,Fee,PhotoAmt,Description,structured_score,bert_similarity,final_score
7084,dcdf98438,Lucas,12,young,male,small,short,1,1,1,1,0,6.0,Lucas is a sturdy little dog. Very well behave...,1.0,0.940067,0.982020
1103,3f222f11f,Rascal,12,young,male,small,short,3,3,3,1,0,1.0,"This cute dog is well behaved, friendly and ta...",1.0,0.939743,0.981923
2281,7564c7b62,Zorro Samdani,12,young,male,small,short,1,1,2,1,0,4.0,"If you want to adopt zorro, you must collect h...",1.0,0.937755,0.981327
7630,1c107ce2b,Zorro,9,young,male,small,short,1,1,2,1,50,4.0,Zorro is a loving and loyal dog adopted from S...,1.0,0.937143,0.981143
6640,bd0a144b0,Mango,9,young,female,small,short,1,1,1,1,200,1.0,My wife got pregnant and therefore we are unab...,1.0,0.930017,0.979005
1695,271bb0de8,Chihuahua,12,young,female,small,short,3,3,3,1,0,2.0,Found this abandoned Chihuahua nearby KFC. I p...,1.0,0.927057,0.978117
7486,b40818da6,Camilo 2,12,young,male,small,short,1,1,1,1,0,1.0,"Very lively dog, as expected from a JR! We are...",1.0,0.922956,0.976887
1986,3af1dedf7,NaN,12,young,female,small,short,3,3,3,2,0,2.0,"I'm not sure of the breed, but this dog has be...",1.0,0.921230,0.976369
454,caf3220f8,Doggie,12,young,male,small,short,1,1,2,2,0,1.0,Hi everyone..I rescued this stray about 2 mont...,1.0,0.918423,0.975527
7367,682b7b7d0,Female Mini Pin,12,young,female,small,short,1,1,2,1,250,1.0,I need to re-home my one year old female Mini ...,1.0,0.917905,0.975372


## 9. Esportazione in CSV

Il file CSV viene salvato nella cartella `data/results`.

Il nome del file contiene l'ID del profilo famiglia scelto.


In [30]:
results_dir = Path("../data/results")
results_dir.mkdir(parents=True, exist_ok=True)

output_file = results_dir / f"ranking_{selected_profile_id}.csv"

full_ranking.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print("File CSV salvato in:")
print(output_file)

print("Numero cani esportati:")
print(len(full_ranking))

File CSV salvato in:
..\data\results\ranking_family_children_apartment.csv
Numero cani esportati:
8132


## 10. Esportazione opzionale della Top 20

Per consultazione rapida viene salvato anche un secondo file con i primi 20 cani consigliati.


In [31]:
top20_file = results_dir / f"top20_{selected_profile_id}.csv"

full_ranking.head(20).to_csv(
    top20_file,
    index=False,
    encoding="utf-8"
)

print("File Top 20 salvato in:")
print(top20_file)

File Top 20 salvato in:
..\data\results\top20_family_children_apartment.csv


## 11. Conclusioni

Questo notebook permette di esportare un ranking completo per uno specifico profilo famiglia.

Il file CSV prodotto contiene tutti i cani valutati dal sistema e permette di analizzare:

- quali cani sono più compatibili;
- quali cani sono stati penalizzati dai vincoli;
- il contributo della compatibilità strutturata;
- il contributo della similarità semantica BERT;
- lo score finale di raccomandazione.

Questo rende il prototipo più utilizzabile e permette di confrontare facilmente i risultati ottenuti per famiglie diverse.
